Split do guia inserido pelo parametro path:str

In [1]:
import io
from typing import List
from pypdf import PdfReader, PdfWriter

def split_pdf(path: str) -> List[bytes]:
    paginas_em_bytes = []
    
    reader = PdfReader(path)
    total_paginas = len(reader.pages)
    
    print(f"[Split] Cortando PDF com {total_paginas} páginas...")
    
    for idx in range(total_paginas):
        writer = PdfWriter()
        writer.add_page(reader.pages[idx])
        
        buffer = io.BytesIO()
        writer.write(buffer)
        

        paginas_em_bytes.append(buffer.getvalue())
        buffer.close()
        
    print(f"[Split] Concluído! {len(paginas_em_bytes)} páginas isoladas em memória.")
    return paginas_em_bytes

Vetorização das páginas do pdf

In [2]:
from typing import List
from google import genai
from google.genai import types
from dotenv import load_dotenv
from langchain_core.embeddings import Embeddings

load_dotenv()

client = genai.Client()


class GeminiEmbeddings(Embeddings):
    def __init__(self, client, model: str = "gemini-embedding-2-preview"):
        self.client = client
        self.model = model

    def _embed(self, contents, task_type: str) -> list[float]:
        resp = self.client.models.embed_content(
            model=self.model,
            contents=contents,
            config=types.EmbedContentConfig(task_type=task_type),
        )
        return resp.embeddings[0].values

    def embed_query(self, text: str) -> List[float]:
        return self._embed(text, task_type="RETRIEVAL_QUERY")

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return [self._embed(t, task_type="RETRIEVAL_DOCUMENT") for t in texts]

    def embed_pdf_bytes(self, page_bytes: bytes) -> List[float]:
        part = types.Part.from_bytes(data=page_bytes, mime_type="application/pdf")
        return self._embed(part, task_type="RETRIEVAL_DOCUMENT")


embeddings = GeminiEmbeddings(client)

Setup do vector store LangChain (Chroma persistente local)

In [3]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="guias_saude",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
    collection_metadata={"hnsw:space": "cosine"},
)

print(f"[Chroma] Coleção pronta. Itens atuais: {vector_store._collection.count()}")

[Chroma] Coleção pronta. Itens atuais: 276


In [4]:
import os
from pypdf import PdfReader
from langchain_core.documents import Document


def extract_page_text(pdf_path: str, page_idx: int) -> str:
    return PdfReader(pdf_path).pages[page_idx].extract_text() or ""


def pipeline(pdf_path: str):
    paginas = split_pdf(pdf_path)
    source_name = os.path.basename(pdf_path)

    documents: list[Document] = []
    embeddings_list: list[list[float]] = []
    ids: list[str] = []

    for i, pagina_bytes in enumerate(paginas):
        num_pagina = i + 1
        print(f"[Pipeline] Vetorizando página {num_pagina}/{len(paginas)}...")

        vetor = embeddings.embed_pdf_bytes(pagina_bytes)
        page_text = extract_page_text(pdf_path, i)

        documents.append(
            Document(
                page_content=page_text,
                metadata={
                    "source": source_name,
                    "path": pdf_path,
                    "page": num_pagina,
                },
            )
        )
        embeddings_list.append(vetor)
        ids.append(f"{source_name}::page_{num_pagina}")

    vector_store._collection.upsert(
        ids=ids,
        documents=[d.page_content for d in documents],
        embeddings=embeddings_list,
        metadatas=[d.metadata for d in documents],
    )
    print(
        f"[Pipeline] {len(ids)} vetores salvos. "
        f"Total na coleção: {vector_store._collection.count()}"
    )


paths = [
    "guias/Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf",
    "guias/Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf",
]

for guia in paths:
    pipeline(guia)

[Split] Cortando PDF com 122 páginas...
[Split] Concluído! 122 páginas isoladas em memória.
[Pipeline] Vetorizando página 1/122...
[Pipeline] Vetorizando página 2/122...
[Pipeline] Vetorizando página 3/122...
[Pipeline] Vetorizando página 4/122...
[Pipeline] Vetorizando página 5/122...
[Pipeline] Vetorizando página 6/122...
[Pipeline] Vetorizando página 7/122...
[Pipeline] Vetorizando página 8/122...
[Pipeline] Vetorizando página 9/122...
[Pipeline] Vetorizando página 10/122...
[Pipeline] Vetorizando página 11/122...
[Pipeline] Vetorizando página 12/122...
[Pipeline] Vetorizando página 13/122...
[Pipeline] Vetorizando página 14/122...
[Pipeline] Vetorizando página 15/122...
[Pipeline] Vetorizando página 16/122...
[Pipeline] Vetorizando página 17/122...
[Pipeline] Vetorizando página 18/122...
[Pipeline] Vetorizando página 19/122...
[Pipeline] Vetorizando página 20/122...
[Pipeline] Vetorizando página 21/122...
[Pipeline] Vetorizando página 22/122...
[Pipeline] Vetorizando página 23/122.

### Exemplo de consulta

In [8]:
def query(pergunta: str, source: str | None = None, k: int = 3):
    filtro = {"source": source} if source else None
    return vector_store.similarity_search_with_score(pergunta, k=k, filter=filtro)


resultados = query(
    "Quais são os sintomas de hipoglicemia?"
)
for doc, score in resultados:
    print(
        f"  score={score:.4f}  {doc.metadata['source']}  "
        f"pág. {doc.metadata['page']}"
    )

  score=0.5800  Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf  pág. 98
  score=0.5813  Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf  pág. 8
  score=0.5943  Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf  pág. 62


In [ ]:
resultados = query(
    "Classificação de risco do pé diabético", # "Diabetes gestacional"

    source="Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf",
)
for doc, score in resultados:
    print(
        f"  score={score:.4f}  {doc.metadata['source']}  "
        f"pág. {doc.metadata['page']}"
    )

# Exemplo extra: usando o vector store como Retriever do LangChain
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
docs = retriever.invoke("Classificação de risco do pé diabético")
print("\n[Retriever]")
for d in docs:
    print(f"  {d.metadata['source']}  pág. {d.metadata['page']}")

  score=0.5162  Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf  pág. 93
  score=0.5221  Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf  pág. 94
  score=0.5416  Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf  pág. 92

[Retriever]
  Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf  pág. 93
  Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf  pág. 94
  Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf  pág. 92


In [17]:
from agent import answer_question

resposta = answer_question(
    vector_store,
    client,
    "Qual a janela ideal de prescricao para tratar o pre-eclampsia?",
    #source="Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf",
    k=5,
)

from IPython.display import Markdown
Markdown(resposta.as_markdown())


As páginas anexadas abordam a **profilaxia** (prevenção) da pré-eclâmpsia, e não a janela ideal de prescrição para o **tratamento** de um caso já estabelecido.

Para a **profilaxia** da pré-eclâmpsia em gestantes com risco aumentado, a prescrição de AAS 100mg tem a seguinte janela:
*   **Duração:** Desde a 12ª semana até a 36ª semana de gestação (pág. 73).
*   **Observações:** Deve ser iniciado preferencialmente antes da 16ª semana, porém há benefícios se iniciado até a 20ª semana. Após essa janela, não há mais evidências de benefício (pág. 73).

As páginas não fornecem uma janela de prescrição ideal para o tratamento da pré-eclâmpsia já diagnosticada.

**Fontes:**
- [Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf — pág. 74](guias/Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf#page=74)  (score=0.4952)
- [Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf — pág. 73](guias/Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf#page=73)  (score=0.5130)
- [Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf — pág. 72](guias/Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf#page=72)  (score=0.5167)
- [Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf — pág. 70](guias/Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf#page=70)  (score=0.5359)
- [Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf — pág. 68](guias/Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf#page=68)  (score=0.5372)